# Supervised Fine-Tuning (SFT)

A practical reference for **supervised fine-tuning (SFT)** — adapting a pretrained language model by continuing training on a labeled dataset of `(prompt, desired-response)` pairs using the same next-token cross-entropy objective the model was pretrained with. SFT is the first alignment step in the canonical post-training stack (**SFT → preference optimization (DPO/RLHF)**) and the workhorse for teaching a base model to follow instructions, adopt a format, or speak a domain's language.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

**Supervised fine-tuning (SFT)** takes a pretrained model and continues training it on a curated set of input→output examples, minimizing the standard **causal language-modeling loss** (token-level cross-entropy) on the *response* tokens. Nothing about the objective changes from pretraining — you are still predicting the next token — but the data is now small, high-quality, and *demonstrative*: every example shows the model exactly what a good answer looks like for a given prompt.

### What is it?

SFT is **behavior cloning for text**. A base model trained on raw web text can complete sentences but doesn't reliably answer questions, follow instructions, or stop talking. SFT shows it thousands of `(instruction, ideal answer)` demonstrations so it learns the *shape* of helpful responses: when to start, what register to use, how to format, and when to stop (the EOS token). The model imitates the demonstrations. This is why SFT is sometimes called **instruction tuning** when the dataset is a broad mix of tasks.

In the alignment pipeline:

```
base model ──SFT──> instruction-following model ──DPO/RLHF──> preference-aligned model
(predicts text)      (follows instructions)                   (prefers good > bad answers)
```

SFT teaches the model *what a good answer looks like*; preference methods then teach it *which of two answers is better*. You almost always do SFT first — preference optimization needs a model that already produces reasonable on-distribution responses.

### Why use it?

- **Instruction following.** Turn a next-token predictor into something that answers questions and obeys system prompts.
- **Format & schema control.** Reliably emit JSON, tool calls, a specific chat template, citations, or a fixed answer structure.
- **Domain & style adaptation.** Teach legal/medical/code register, a brand voice, or a low-resource language the base model handles poorly.
- **Capability distillation.** Fine-tune a small open model on outputs from a larger one to recover much of its behavior cheaply.
- **Cheaper than RLHF.** SFT is plain supervised learning — no reward model, no sampling loop, no PPO. It is stable, fast, and easy to debug. With LoRA it runs on a single consumer GPU.

### When to use it?

- You have (or can synthesize/curate) **hundreds-to-thousands of high-quality demonstrations** of the behavior you want.
- The task is about **how to respond** (format, tone, following instructions), not a numeric metric you can only express as a *preference* between outputs.
- Prompting alone is unreliable, too verbose, or too expensive at inference, and you want the behavior baked into weights.

### When *not* to use it

- The knowledge is **dynamic or factual lookup** — use RAG; SFT is poor at injecting fresh facts and prone to making the model confidently hallucinate them.
- You only have **preference/ranking signal** (A is better than B) rather than gold demonstrations — that's DPO/RLHF territory (still do a small SFT warm-up first).
- A few-shot prompt already works well enough — don't pay training and serving complexity for marginal gains.

## Key Features

### Core concepts and the levers you actually tune

| Concept | Description | Why it matters |
|---------|-------------|----------------|
| **CLM / cross-entropy loss** | Same next-token objective as pretraining, applied to the response | No new loss to implement; SFT is "more pretraining, better data" |
| **Completion-only / loss masking** | Compute loss on *response* tokens only; mask the prompt to `-100` | The model shouldn't be graded on reproducing the user's question; masking prompts measurably improves quality |
| **Chat template** | The exact special-token formatting (`<|user|>`, `<|assistant|>`, …) wrapping each turn | Train/serve mismatch here silently destroys quality — must match what you serve with |
| **Packing** | Concatenate many short examples into one max-length sequence | 2–5× throughput by eliminating padding waste |
| **PEFT / LoRA** | Freeze base weights; train small low-rank adapters | ~0.1–1% of params trained; fits 7B on one 24 GB GPU; adapters are swappable |
| **Epochs (1–3)** | Number of passes over the SFT set | SFT memorizes fast; >3 epochs usually overfits and degrades generalization |
| **Learning rate** | Typically `1e-5`–`2e-5` full FT, `1e-4`–`3e-4` for LoRA | The single most important knob; too high causes the model to "forget" and produce garbage |
| **EOS token** | The stop token appended to every demonstration | Forget it and the model never learns to stop generating |

## Architecture Overview

SFT is a **single supervised training job**, not a system. The interesting structure is the **data pipeline** that turns raw demonstrations into masked, templated, packed token tensors, plus the choice of *full fine-tuning vs. LoRA*.

```
 raw demos                template + tokenize          batch                forward/backward
 [{messages:[...]}]  ──>   apply_chat_template    ──>   pack to max_len  ──>  base model (frozen?)
        │                  mask prompt tokens=-100      pad/attention mask         │
        │                                                                          ▼
        └────────────────────────────────────────────────────────────────  + LoRA adapters (trainable)
                                                                                   │
                                                          cross-entropy on response tokens only
                                                                                   │
                                                                                   ▼
                                                              save: full checkpoint  OR  adapter (~tens of MB)
```

### Components

1. **Dataset of demonstrations.** Each example is a conversation (`messages`) or a `(prompt, completion)` pair. Quality and diversity dominate quantity — 1k clean examples beat 100k noisy ones.
2. **Chat template + tokenizer.** Renders messages into the exact string the served model expects, inserts role/turn special tokens, and tokenizes. The same template must be used at inference.
3. **Loss masking (completion-only collator).** Sets label `-100` on prompt/system tokens so they don't contribute to the loss — the model is trained to *produce* answers, not echo prompts.
4. **The model + (optional) LoRA.** Either every weight is trainable (full FT, needs ~16× model-size GPU memory with Adam states) or the base is frozen and small low-rank adapters carry the update (LoRA/QLoRA).
5. **Trainer loop.** AdamW, a short warmup + cosine/linear decay, gradient checkpointing, mixed precision (bf16). Usually `TRL`'s `SFTTrainer` or a plain HF `Trainer`.
6. **Output.** A merged checkpoint (full FT) or a tiny adapter you load on top of the base at serve time.

## Installation

### Prerequisites

- Python 3.9+
- NumPy — the self-contained demos below run on CPU with no ML framework.
- For real LLM SFT: PyTorch + an NVIDIA GPU. A 7–8B model needs ≈16 GB with QLoRA, ≈24 GB with bf16 LoRA, and ≈80 GB+ for full fine-tuning (optimizer states dominate).
- The Hugging Face stack: `transformers`, `trl` (`SFTTrainer`), `peft` (LoRA), `datasets`, `accelerate`, and `bitsandbytes` for 4-bit QLoRA.

In [ ]:
# The NumPy demos below need nothing extra. For real supervised fine-tuning:
# %pip install -U "transformers>=4.40" "trl>=0.9" peft accelerate datasets
# For 4-bit QLoRA (train a 7B on a single 16 GB GPU):
# %pip install -U bitsandbytes
# Optional experiment tracking / faster training:
# %pip install -U wandb flash-attn --no-build-isolation

## Basic Usage

### Quick Start Example

The entire idea of SFT — minimizing next-token cross-entropy on demonstrations, **but only on the response tokens** — fits in a few lines of pure NumPy. Below we build a toy model over a tiny vocabulary and show the two things that make SFT *SFT*: (1) the loss is plain cross-entropy, and (2) **prompt tokens are masked out** so the model is graded only on what it should generate.

In [ ]:
# SFT distilled to pure NumPy: cross-entropy on the next token, masked to the response.
import numpy as np

rng = np.random.default_rng(0)

# Tiny vocab. Treat "PROMPT" tokens as ids 0-3, "RESPONSE" tokens as ids 4-7, EOS=8.
VOCAB = 9
PROMPT_IDS, RESP_IDS, EOS = {0, 1, 2, 3}, {4, 5, 6, 7}, 8

# One demonstration: prompt tokens, then response tokens, then EOS.
seq    = np.array([0, 1, 2,   4, 5, 6, 7, 8])          # full sequence
labels = np.array([-100, -100, -100,  4, 5, 6, 7, 8])  # -100 = "don't compute loss here"
# ^ the prompt (0,1,2) is masked: we never grade the model for reproducing the question.

# A toy "model": logits = learnable score table indexed by the *previous* token.
# (A real model conditions on the whole prefix; this captures the loss mechanics.)
W = rng.normal(0, 0.1, size=(VOCAB, VOCAB))  # W[prev] -> logits over next token

def softmax(z):
    z = z - z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)

def sft_loss_and_grad(W, seq, labels):
    """Token-level cross-entropy over next-token predictions, masked by labels."""
    loss, gradW, n = 0.0, np.zeros_like(W), 0
    for t in range(len(seq) - 1):
        target = labels[t + 1]            # predict token t+1 from token t
        if target == -100:                # masked prompt token -> skip
            continue
        p = softmax(W[seq[t]])            # distribution over next token
        loss += -np.log(p[target] + 1e-12)
        g = p.copy(); g[target] -= 1.0    # dL/dlogits for cross-entropy
        gradW[seq[t]] += g
        n += 1
    return loss / n, gradW / n

# Train: a few SGD steps drive loss down and the model learns the response sequence.
lr = 1.0
for step in range(400):
    loss, g = sft_loss_and_grad(W, seq, labels)
    W -= lr * g
print(f"final masked CE loss: {loss:.4f}")

# Greedy-decode the response given the prompt's last token -> should reproduce 4,5,6,7,8.
out, prev = [], 2
for _ in range(5):
    prev = int(softmax(W[prev]).argmax())
    out.append(prev)
print("decoded response:", out, "(target was [4, 5, 6, 7, 8] ending in EOS=8)")

### The real-world shape

In production you don't hand-roll the loss — you hand a dataset of chat messages to `TRL`'s `SFTTrainer`, which applies the tokenizer's **chat template**, masks the prompt with a completion-only collator, packs sequences, and runs AdamW. The snippet below is the canonical recipe; it's gated behind a `try/except` so this notebook still runs without a GPU or the HF stack installed.

In [ ]:
# Canonical SFT recipe with TRL + LoRA. Illustrative: gated so the notebook runs anywhere.
try:
    import torch
    from datasets import load_dataset
    from peft import LoraConfig
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import SFTConfig, SFTTrainer

    model_id = "meta-llama/Llama-3.2-1B"   # small enough for a single modest GPU

    tok = AutoTokenizer.from_pretrained(model_id)
    tok.pad_token = tok.eos_token          # most base models lack a pad token

    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.bfloat16, device_map="auto"
    )

    # Dataset must expose a "messages" column: [{"role": "user", ...}, {"role": "assistant", ...}]
    train_ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft[:2000]")

    # LoRA: freeze the base, train ~0.5% of params as low-rank adapters.
    peft_cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )

    cfg = SFTConfig(
        output_dir="sft-llama-1b",
        num_train_epochs=2,                 # 1-3 epochs is plenty for SFT
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,      # effective batch = 32
        learning_rate=2e-4,                 # LoRA tolerates higher LR than full FT (~2e-5)
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        bf16=True,
        gradient_checkpointing=True,        # trade compute for memory
        packing=True,                       # concatenate short examples -> big throughput win
        max_seq_length=2048,
        logging_steps=10,
        save_strategy="epoch",
    )

    trainer = SFTTrainer(
        model=model, args=cfg, train_dataset=train_ds,
        processing_class=tok, peft_config=peft_cfg,
    )
    trainer.train()
    trainer.save_model()   # saves the LoRA adapter (tens of MB), not the full model
except Exception as e:
    print(f"[illustrative only — not executed here] {type(e).__name__}: {e}")

## Advanced Features

### Prompt masking, packing, and LoRA — the three knobs that matter

**1. Completion-only loss (prompt masking).** By default a language model computes loss on *every* token. For SFT you usually want loss only on the assistant's reply, so the model isn't rewarded for memorizing prompts. `TRL` does this with `DataCollatorForCompletionOnlyLM`, which finds the response template (e.g. the `<|assistant|>` marker) and sets every label before it to `-100`. The NumPy demo above showed the mechanic; here's the production form.

**2. Packing.** SFT datasets are full of short examples. Padding each to `max_seq_length` wastes most of the GPU. **Packing** concatenates examples (separated by EOS) into dense `max_len` blocks, often 2–5× throughput. The cost: examples can bleed across the attention boundary unless you use a packing implementation that resets attention per example.

**3. LoRA / QLoRA.** Instead of updating the weight matrix `W`, LoRA learns a low-rank update `ΔW = B·A` (with `A ∈ ℝ^{r×d}`, `B ∈ ℝ^{d×r}`, `r≪d`) and keeps `W` frozen. You train `r/d` of the parameters, the optimizer state shrinks proportionally, and the result is a tiny swappable adapter. **QLoRA** additionally quantizes the frozen base to 4-bit, letting a 7B model train on a 16 GB GPU. The demo below shows *why* a rank-`r` update is so cheap yet expressive.

In [ ]:
# Why LoRA is cheap: a rank-r update B@A approximates a full dxd weight change with far fewer params.
import numpy as np
rng = np.random.default_rng(1)

d, r = 256, 8
full_update = rng.normal(0, 1, size=(d, d))     # what full fine-tuning would change: d*d params
A = rng.normal(0, 1, size=(r, d))               # LoRA factors
B = rng.normal(0, 1, size=(d, r))
lora_update = B @ A                              # rank-r approximation: 2*d*r params

full_params = d * d
lora_params = 2 * d * r
print(f"full-FT trainable params for this matrix: {full_params:,}")
print(f"LoRA (r={r}) trainable params:            {lora_params:,}  "
      f"({100*lora_params/full_params:.1f}% of full)")

# A real weight *delta* from fine-tuning is typically low-rank, so a small r captures most of it.
# Demonstrate: a genuinely low-rank target is reconstructed almost perfectly by rank-r LoRA.
true_low_rank = (rng.normal(size=(d, r)) @ rng.normal(size=(r, d)))   # exact rank r
U, S, Vt = np.linalg.svd(true_low_rank)
best_rank_r = (U[:, :r] * S[:r]) @ Vt[:r]                              # optimal rank-r fit
rel_err = np.linalg.norm(true_low_rank - best_rank_r) / np.linalg.norm(true_low_rank)
print(f"rank-{r} reconstruction error of a rank-{r} target: {rel_err:.2e}  (~0 -> LoRA suffices)")

### The chat template: the bug that silently halves your quality

The most common *invisible* SFT failure is a **train/serve template mismatch**. The model learns to respond to one exact byte sequence of special tokens; if you serve with a different template (or none), quality collapses for reasons that never show up in the training loss. Always render with the tokenizer's own `apply_chat_template` and verify it matches your serving stack.

In [ ]:
# Inspect exactly what the model will be trained on. Mismatch here is the #1 silent SFT bug.
messages = [
    {"role": "system", "content": "You are a terse assistant."},
    {"role": "user", "content": "What is supervised fine-tuning?"},
    {"role": "assistant", "content": "Training a model on labeled prompt-response demos."},
]

def fake_apply_chat_template(messages, add_generation_prompt=False):
    """Stand-in for tokenizer.apply_chat_template so this cell runs without transformers.
    Real code: tok.apply_chat_template(messages, tokenize=False)."""
    roles = {"system": "<|system|>", "user": "<|user|>", "assistant": "<|assistant|>"}
    out = "<|begin_of_text|>"
    for m in messages:
        out += f"{roles[m['role']]}\n{m['content']}<|eot_id|>\n"
    if add_generation_prompt:
        out += "<|assistant|>\n"
    return out

rendered = fake_apply_chat_template(messages)
print(rendered)
print("--- response template the completion-only collator masks up to: '<|assistant|>' ---")
print("Train and serve MUST use this identical template, including the trailing EOS/<|eot_id|>.")

## Use Cases

### Real-world applications of supervised fine-tuning

#### Use Case 1: Instruction tuning a base model

- **Context:** You have a strong *base* model (predicts text) but need a chat assistant that follows instructions.
- **Implementation:** SFT on a broad instruction mix (e.g. UltraChat, OpenHermes, or your own curated set) covering many task types, formats, and refusals.
- **Results:** The base model becomes a usable assistant — answers questions, follows system prompts, stops at EOS. This is the SFT stage of every open chat model (Llama-Instruct, Qwen-Instruct, etc.).

#### Use Case 2: Structured-output / tool-calling reliability

- **Context:** Your agent must emit valid JSON or function calls 100% of the time; prompting gets you to ~95% and the failures are expensive.
- **Implementation:** SFT on a few thousand `(request, exact-JSON-or-tool-call)` pairs. Mask the prompt; train 1–2 epochs; validate every output parses.
- **Results:** Schema-valid output rate climbs toward 100%, shorter prompts (no need for long format instructions), lower latency and token cost at inference.

#### Use Case 3: Domain & capability distillation

- **Context:** A large frontier model handles your niche (legal summarization, a low-resource language, your codebase's conventions) but is too costly to serve at scale.
- **Implementation:** Generate high-quality demonstrations with the large model, filter them, and SFT a small open model (1–8B) with LoRA on the distilled set.
- **Results:** The small model recovers much of the behavior at a fraction of the serving cost; the adapter is tiny and swappable per domain.

## Best Practices

### Recommended practices for supervised fine-tuning

1. **Data quality beats data quantity, by a lot.** A thousand clean, diverse, correctly-formatted demonstrations outperform tens of thousands of noisy ones. Manually read a random sample of your data — you will find problems. Deduplicate and decontaminate against your eval set.
2. **Mask the prompt (completion-only loss).** Train on response tokens only. It consistently improves quality and prevents the model from wasting capacity reproducing inputs.
3. **Use the model's own chat template, end to end.** Render training data with `apply_chat_template` and serve with the identical template. Verify the rendered string by eye once — it catches the most damaging silent bug.
4. **Always append EOS.** Every demonstration must end with the stop token, or the model never learns to stop and rambles at inference.
5. **Few epochs, modest LR.** Start at 1–3 epochs. Use `~2e-5` for full FT, `~1e-4`–`3e-4` for LoRA. SFT memorizes quickly; more epochs usually *hurt* generalization.
6. **Prefer LoRA/QLoRA unless you have a reason not to.** Cheaper, faster, regularizing, and the adapters are swappable. Reach for full FT only when LoRA demonstrably underfits the behavior you need.
7. **Hold out a real eval set and judge on generations, not loss.** Validation loss can drop while behavior degrades. Track task-specific metrics and read sample outputs every checkpoint.

## Common Pitfalls

### What to avoid when using supervised fine-tuning

1. **Train/serve template mismatch.** Training with one chat template and serving with another (or raw text) silently tanks quality with no signal in the loss curve. *Avoid:* always use the tokenizer's `apply_chat_template` and assert your serving path produces byte-identical formatting.
2. **Forgetting to mask the prompt.** Computing loss on prompt tokens wastes capacity and weakens instruction-following. *Avoid:* use a completion-only collator / response template.
3. **Missing EOS token.** Without it the model never learns to stop and generates until `max_new_tokens`. *Avoid:* confirm every example ends with EOS and that `pad_token` is set (often to `eos_token`) for batching.
4. **Overfitting / catastrophic forgetting.** Too many epochs or too high an LR makes the model memorize your set and lose general ability (it "forgets" how to do anything but your task). *Avoid:* 1–3 epochs, LoRA for regularization, and a held-out eval to catch the turn.
5. **Expecting SFT to inject facts.** SFT teaches *behavior*, not a reliable knowledge store. Fine-tuning on facts often makes the model hallucinate them more confidently. *Avoid:* use RAG for knowledge; reserve SFT for format, style, and instruction-following.
6. **Noisy or homogeneous data.** Garbage, near-duplicates, or a single task type produce a brittle, narrow model. *Avoid:* curate, deduplicate, and diversify; read a sample by hand.

## Performance Optimization

### Optimizing supervised fine-tuning for production

The two scarce resources are **GPU memory** and **wall-clock throughput**. The standard levers:

#### Configuration tuning

- **LoRA / QLoRA** — train ~0.1–1% of parameters; 4-bit quantization of the frozen base cuts memory ~4×. Biggest single win for fitting larger models.
- **Packing** (`packing=True`) — eliminate padding waste by concatenating short examples; often 2–5× throughput on instruction data.
- **Gradient checkpointing** — recompute activations in the backward pass instead of storing them: large memory savings for ~20–30% extra compute. Essential for long sequences.
- **bf16 mixed precision** + **Flash Attention 2** — faster and more memory-efficient attention; bf16 is the safe default on Ampere+ GPUs.
- **Gradient accumulation** — reach a large *effective* batch size without the memory of a large physical batch.
- **Sequence length** — train at the shortest `max_seq_length` that fits your data; memory and compute scale with it.

The cell below estimates the GPU memory full fine-tuning vs. LoRA actually demand, which is the calculation that decides your hardware.

In [ ]:
# Memory math: why LoRA/QLoRA changes what hardware you need.
# AdamW stores, per *trainable* parameter: the param + grad + 2 optimizer moments.
def mem_gb(n_params, bytes_per_trainable, frozen_base_bytes):
    trainable = n_params * bytes_per_trainable          # param+grad+moments for trainable weights
    frozen    = n_params * frozen_base_bytes            # frozen base just needs forward storage
    return (trainable + frozen) / 1e9

P = 7e9   # 7B model

# Full FT in bf16 + AdamW: 2 (param) + 2 (grad) + 8 (fp32 moments) = ~16 bytes / param, all trainable.
full = mem_gb(P, bytes_per_trainable=16, frozen_base_bytes=0)

# LoRA bf16: base frozen (2 bytes, fwd only); only ~0.5% of params carry the 16-byte optimizer cost.
trainable_frac = 0.005
lora = (P * trainable_frac * 16 + P * 2) / 1e9

# QLoRA: base quantized to 4-bit (~0.5 bytes/param) + the same tiny trainable adapters.
qlora = (P * trainable_frac * 16 + P * 0.5) / 1e9

print(f"7B full fine-tune (bf16 + AdamW): ~{full:5.0f} GB  -> multi-GPU / A100-80GB territory")
print(f"7B LoRA (bf16 base, r small):     ~{lora:5.0f} GB  -> fits a single 24 GB GPU")
print(f"7B QLoRA (4-bit base):            ~{qlora:5.0f} GB  -> fits a single 16 GB GPU")
print("\n(Excludes activations/KV cache, which gradient checkpointing + short seq_len keep in check.)")

## Production Deployment

### Deploying supervised fine-tuning in production

SFT is a **training-time** activity: you run the job offline, produce a checkpoint or adapter, and then deploy that artifact behind an inference server (vLLM, TGI, TensorRT-LLM, …). Two artifact shapes:

- **Full fine-tune** → a complete model checkpoint you serve directly.
- **LoRA adapter** → a tens-of-MB file you either (a) **merge** into the base for a standalone model, or (b) load **dynamically** on top of a shared base — vLLM and TGI can hot-swap multiple adapters over one base model, which is ideal for multi-tenant serving.

#### Docker: a reproducible SFT training job

```dockerfile
# Train, then exit. Run on a GPU node; mount data in, checkpoints out.
FROM nvidia/cuda:12.4.1-cudnn-runtime-ubuntu22.04
RUN apt-get update && apt-get install -y python3-pip git && rm -rf /var/lib/apt/lists/*
RUN pip3 install --no-cache-dir \
    "torch" "transformers>=4.40" "trl>=0.9" peft accelerate datasets bitsandbytes
WORKDIR /workspace
COPY train_sft.py .
# Data in /data, adapter out to /checkpoints (both mounted volumes).
ENTRYPOINT ["accelerate", "launch", "train_sft.py", \
            "--dataset", "/data/sft.jsonl", "--output_dir", "/checkpoints"]
```

#### Kubernetes: SFT as a GPU batch Job

```yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: sft-llama-lora
spec:
  backoffLimit: 2                 # training jobs should not retry forever
  template:
    spec:
      restartPolicy: Never
      containers:
        - name: trainer
          image: registry.example.com/sft-trainer:llama-1b
          args: ["--epochs", "2", "--lr", "2e-4", "--lora_r", "16"]
          resources:
            limits:
              nvidia.com/gpu: 1   # one GPU; bump + accelerate config for multi-GPU
          volumeMounts:
            - { name: data, mountPath: /data }
            - { name: ckpt, mountPath: /checkpoints }
      volumes:
        - { name: data, persistentVolumeClaim: { claimName: sft-data } }
        - { name: ckpt, persistentVolumeClaim: { claimName: sft-checkpoints } }
      nodeSelector:
        cloud.google.com/gke-accelerator: nvidia-l4
```

#### Serving a LoRA adapter dynamically (vLLM)

```bash
# Serve the base once; attach the SFT adapter by name at request time.
vllm serve meta-llama/Llama-3.2-1B   --enable-lora   --lora-modules my-sft=/checkpoints/adapter

# Requests select the adapter via the "model" field:
curl localhost:8000/v1/chat/completions   -d '{"model": "my-sft", "messages": [{"role": "user", "content": "hi"}]}'
```

## Monitoring and Observability

### Monitoring supervised fine-tuning in production

#### Key metrics to track

- **Training & validation loss** — should fall smoothly. Watch for **val loss rising while train loss falls** (overfitting); that's your epoch cap signal. Loss alone is *not* sufficient — pair it with generation evals.
- **Token accuracy / perplexity on a held-out set** — a cheap proxy that correlates with behavior.
- **Task-specific eval on generations** — the metric that actually matters: JSON-valid rate, exact-match, win-rate vs. the base via an LLM judge, format-compliance, refusal correctness. Run it every checkpoint.
- **Gradient & weight norms** — spikes signal an LR that's too high or bad data; a flat-zero grad norm signals a masking/template bug (nothing is being learned).
- **Throughput** (tokens/sec) and **GPU memory/utilization** — for cost and to confirm packing/checkpointing are working.
- **Catastrophic-forgetting check** — eval on a few *general* benchmarks (not just your task) to confirm you didn't break base capabilities.

#### Logging best practices

- Log the **full effective config** (model, LR, epochs, LoRA rank, seed, data hash) with every run so results are reproducible and comparable — use W&B / MLflow / TensorBoard.
- Log a fixed set of **sample generations** at each eval step; reading 10 outputs catches failures no scalar metric will.
- Version the **dataset** (hash + row count) alongside the checkpoint — "which data produced this model?" must always be answerable.

## Troubleshooting

### Common issues with supervised fine-tuning

#### Issue 1: The fine-tuned model produces garbage / repeats / never stops

**Symptoms**: Output is incoherent, loops, or runs to `max_new_tokens` without stopping.

**Cause**: Almost always a **template or EOS problem** — train/serve template mismatch, or demonstrations weren't terminated with the EOS token so the model never learned to stop.

**Solution**: Render training data with `apply_chat_template`, confirm each example ends in EOS, set `pad_token`, and verify the serving path uses the identical template byte-for-byte.

#### Issue 2: Loss goes down but the model got *worse* at everything else

**Symptoms**: Your task metric improves slightly while general ability collapses (can't do unrelated tasks it used to handle).

**Cause**: **Catastrophic forgetting / overfitting** — LR too high, too many epochs, or data too narrow.

**Solution**: Lower the LR, cut to 1–2 epochs, switch to LoRA (it perturbs the base less), diversify the data, and add a general-capability eval to catch the regression early.

#### Issue 3: Out-of-memory during training

**Symptoms**: CUDA OOM at start of training or on the first long batch.

**Cause**: Full optimizer states and/or activations exceed GPU memory — common when attempting full FT or long sequences on a single GPU.

**Solution**: Enable gradient checkpointing, switch to LoRA/QLoRA, reduce `per_device_train_batch_size` (raise `gradient_accumulation_steps` to keep the effective batch), shorten `max_seq_length`, and use bf16.

#### Issue 4: Grad norm is zero / loss doesn't move

**Symptoms**: Loss is flat from step 0; gradient norm ≈ 0.

**Cause**: Every label was masked (response template not found, so the collator masked *everything*), or the LoRA target modules don't exist on this architecture.

**Solution**: Print a tokenized example and confirm some labels are not `-100`; verify the response template string actually appears in the rendered text; check `target_modules` matches the model's layer names.

## Comparison with Alternatives

### How supervised fine-tuning compares to other approaches

| Dimension | Supervised fine-tuning (SFT) | Prompting / few-shot | RAG | DPO / RLHF | Continued pretraining |
|-----------|------------------------------|----------------------|-----|------------|-----------------------|
| **Signal needed** | Gold `(prompt, response)` demos | None (just a prompt) | A document corpus | Preference pairs (A≻B) | Large unlabeled corpus |
| **Teaches** | *How* to respond (behavior, format) | Nothing persistent | Fresh facts at query time | *Which* response is preferred | Broad domain knowledge |
| **Changes weights?** | Yes | No | No | Yes (warm-started from SFT) | Yes |
| **Cost** | Low–moderate (esp. with LoRA) | ~free | Low (retrieval infra) | High (reward model / sampling) | Very high |
| **Best for** | Instruction-following, format, style, distillation | Quick iteration, prototyping | Dynamic/factual knowledge | Nuanced quality, safety, refusals | New domain/language from scratch |
| **Hallucination risk** | Can *worsen* if used for facts | Inherits base | Reduces (grounded) | Reduces (with good data) | Neutral |

### When to choose SFT

- You have **demonstrations of the desired behavior** and want it baked into the weights for cheaper, faster, more reliable inference than prompting.
- The goal is **format, style, or instruction-following** — not injecting facts (use RAG) and not optimizing a preference you can only express as a ranking (use DPO/RLHF, after an SFT warm-up).
- You want the **simplest, most stable** training method that moves model behavior — SFT is plain supervised learning and should be your first post-training step before reaching for anything fancier.

## Resources

### Official documentation

- TRL `SFTTrainer` (the standard SFT API): https://huggingface.co/docs/trl/en/sft_trainer
- Hugging Face PEFT / LoRA: https://huggingface.co/docs/peft/en/index
- Transformers training & fine-tuning guide: https://huggingface.co/docs/transformers/en/training

### Tutorials and guides

- Hugging Face — Supervised Fine-Tuning chapter (LLM course): https://huggingface.co/learn/llm-course/en/chapter11/1
- QLoRA: Efficient Finetuning of Quantized LLMs (Dettmers et al., 2023): https://arxiv.org/abs/2305.14314
- LoRA: Low-Rank Adaptation of Large Language Models (Hu et al., 2021): https://arxiv.org/abs/2106.09685
- LIMA: Less Is More for Alignment (Zhou et al., 2023) — why data *quality* dominates: https://arxiv.org/abs/2305.11206

### Community resources

- TRL GitHub (issues, recipes, examples): https://github.com/huggingface/trl
- vLLM LoRA serving docs (deploy adapters dynamically): https://docs.vllm.ai/en/latest/features/lora.html
- Hugging Face forums — fine-tuning category: https://discuss.huggingface.co/c/fine-tuning/

### Related technologies

- **DPO / RLHF** — the preference-optimization step that follows SFT.
- **PEFT / LoRA / QLoRA** — parameter-efficient methods that make SFT cheap.
- **RAG** — the alternative when the goal is knowledge, not behavior.
- **Instruction tuning** — SFT on a broad, multi-task instruction dataset.